# AI Music OS — F5-TTS Voice Clone V5

Free local models; no API key. Choose **T4 GPU**, then **Run all** in a fresh Colab session. Open the printed Gradio link. Each notebook uses its own pinned Python 3.11 environment, leaving Colab's Python 3.13 packages untouched. Models and WAV outputs remain on Drive. Keep the notebook session open; free Colab can disconnect it.

Upload this notebook directly to Colab, or replace its matching file in GitHub. No other repository files are required. Dependency and CPU import tests passed locally; Colab T4 inference is not certified.


In [ ]:
MODULE = 'f5'
# Separate Python 3.11 environment; do not install AI packages into Colab's kernel.
from google.colab import drive
from pathlib import Path
import os, sys, subprocess, time, re
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/AI_Music_OS')
CACHE = ROOT / 'cache' / 'huggingface'
OUT = ROOT / 'outputs' / {'ace': 'ace_step', 'kokoro': 'kokoro', 'f5': 'qwen_voice'}[MODULE]
VOICE_BANK = ROOT / 'voice_bank'
MODELS = ROOT / 'models' / 'ace_step'
for p in (CACHE, OUT, VOICE_BANK, MODELS): p.mkdir(parents=True, exist_ok=True)
ENV = os.environ.copy()
ENV['HF_HOME'] = str(CACHE)
ENV['HUGGINGFACE_HUB_CACHE'] = str(CACHE / 'hub')
ENV.pop('TRANSFORMERS_CACHE', None)
ENV['PYTHONUNBUFFERED'] = '1'
WORK = Path('/content/ai_music_v5') / MODULE
WORK.mkdir(parents=True, exist_ok=True)
LOG = OUT / 'install_v5.log'
def run_logged(cmd, cwd=None):
    with LOG.open('a') as log:
        result = subprocess.run(cmd, cwd=cwd, env=ENV, stdout=log, stderr=subprocess.STDOUT)
    if result.returncode:
        print(LOG.read_text(errors='replace')[-18000:])
        raise RuntimeError(f'Command failed ({result.returncode}). Complete output: {LOG}')
UV_HOME = Path('/content/ai_music_v5/bootstrap')
UV = UV_HOME / 'bin' / 'uv'
if not UV.exists():
    run_logged([sys.executable, '-m', 'venv', str(UV_HOME)])
    run_logged([str(UV_HOME / 'bin' / 'python'), '-m', 'pip', 'install', '--no-cache-dir', 'uv==0.13.8'])
run_logged([str(UV), 'python', 'install', '3.11.18'])
VENV = WORK / 'venv'
PY = VENV / 'bin' / 'python'
if not PY.exists(): run_logged([str(UV), 'venv', '--python', '3.11.18', str(VENV)])
print('Colab kernel:', sys.version.split()[0], '| AI runtime: Python 3.11.18')
print('Outputs:', OUT, '| Installation log:', LOG)


In [ ]:
LOCK_CONTENT = 'accelerate==1.15.0\naiofiles==23.2.1\naiohappyeyeballs==2.7.1\naiohttp==3.14.5\naiosignal==1.4.0\nannotated-doc==0.0.6\nannotated-types==0.8.0\nantlr4-python3-runtime==4.9.3\nanyio==4.16.0\nattrs==26.1.0\naudioread==3.1.0\nbitsandbytes==0.50.2\nboto3==1.44.3\nbotocore==1.44.3\ncached-path==1.8.10\ncertifi==2026.10.23\ncffi==2.1.1\ncharset-normalizer==3.5.2\nclick==8.5.1\ncloudpickle==3.1.2\ncontourpy==1.3.3\ncryptography==50.0.2\ncwevals==0.0.1b1\ncycler==0.12.1\ndatasets==3.4.1\ndecorator==5.3.1\ndill==0.3.8\ndistro==1.9.0\neinops==0.8.2\neinx==0.4.3\nema-pytorch==0.8.3\nencodec==0.1.1\nf5-tts==1.1.10\nfastapi==0.162.7\nffmpy==1.0.0\nfilelock==3.32.7\nfonttools==4.68.0\nfrozendict==2.5.1\nfrozenlist==1.8.0\nfsspec==2024.12.0\ngoogle-api-core==2.44.0\ngoogle-auth==2.62.2\ngoogle-cloud-core==2.8.0\ngoogle-cloud-storage==3.17.1\ngoogle-crc32c==1.9.0\ngoogle-resumable-media==2.11.0\ngoogleapis-common-protos==1.76.0\ngradio==5.22.0\ngradio-client==1.8.0\ngroovy==0.1.2\nh11==0.16.0\nhttpcore==1.0.9\nhttpx==0.28.1\nhuggingface-hub==0.30.2\nhydra-core==1.3.8\nidna==3.20\njinja2==3.1.6\njmespath==1.1.0\njoblib==1.6.0\nkiwisolver==1.5.1\nlazy-loader==0.6\nlibrosa==0.11.0\nllvmlite==0.50.0\nloguru==0.8.0\nmarkdown-it-py==4.3.0\nmarkupsafe==3.0.4\nmatplotlib==3.11.2\nmdurl==0.1.2\nmpmath==1.3.0\nmsgpack==1.2.4\nmultidict==7.2.1\nmultiprocess==0.70.16\nnarwhals==2.28.0\nnetworkx==3.6.1\nnumba==0.68.1\nnumpy==1.26.4\nnvidia-cublas-cu12==12.4.5.8\nnvidia-cuda-cupti-cu12==12.4.127\nnvidia-cuda-nvrtc-cu12==12.4.127\nnvidia-cuda-runtime-cu12==12.4.127\nnvidia-cudnn-cu12==9.1.0.70\nnvidia-cufft-cu12==11.2.1.3\nnvidia-curand-cu12==10.3.5.147\nnvidia-cusolver-cu12==11.6.1.9\nnvidia-cusparse-cu12==12.3.1.170\nnvidia-cusparselt-cu12==0.6.2\nnvidia-nccl-cu12==2.21.5\nnvidia-nvjitlink-cu12==12.4.127\nnvidia-nvtx-cu12==12.4.127\nomegaconf==2.3.1\nopentelemetry-api==1.45.2\nopentelemetry-exporter-http-transport==0.66b2\nopentelemetry-exporter-otlp-common==0.66b2\nopentelemetry-exporter-otlp-proto-common==1.45.2\nopentelemetry-exporter-otlp-proto-http==1.45.2\nopentelemetry-proto==1.45.2\nopentelemetry-sdk==1.45.2\nopentelemetry-semantic-conventions==0.66b2\norjson==3.13.0\npackaging==26.3\npandas==2.3.3\npillow==11.3.0\npip==25.1.1\nplatformdirs==4.14.5\npooch==1.9.0\npropcache==0.5.4\nproto-plus==1.29.0\nprotobuf==7.36.2\npsutil==7.2.2\npyarrow==19.0.1\npyasn1==0.6.5\npyasn1-modules==0.4.2\npycparser==3.12\npydantic==2.10.6\npydantic-core==2.27.2\npydub==0.25.1\npygments==2.21.0\npyparsing==3.3.3\npypinyin==0.55.0\npython-dateutil==2.9.0.post0\npython-multipart==0.0.33\npytz==2026.6\npyyaml==6.0.3\nregex==2026.11.9\nrequests==2.34.2\nrich==13.9.4\nrjieba==0.2.2\nruff==0.17.5\ns3transfer==0.19.3\nsafehttpx==0.1.7\nsafetensors==0.8.0\nscikit-learn==1.9.1\nscipy==1.13.1\nsemantic-version==2.10.0\nshellingham==1.5.4\nsix==1.17.0\nsniffio==1.3.1\nsoundfile==0.13.1\nsoxr==1.1.0\nstarlette==0.52.1\nsympy==1.13.1\nthreadpoolctl==3.8.0\ntokenizers==0.21.4\ntomli==2.5.0\ntomlkit==0.13.3\ntorch==2.6.0+cu124\ntorchaudio==2.6.0+cu124\ntorchcodec==0.2.0\ntorchdiffeq==0.2.5\ntorchvision==0.21.0+cu124\ntqdm==4.70.1\ntransformers==4.50.0\ntransformers-stream-generator==0.0.5\ntriton==3.2.0\ntyper==0.29.1\ntyping-extensions==4.16.0\ntyping-inspection==0.4.5\ntzdata==2026.6\nunidecode==1.4.0\nurllib3==2.8.0\nuvicorn==0.54.0\nvocos==0.1.0\nwandb==0.31.0\nwebsockets==15.0.1\nx-transformers==1.44.8\nxxhash==4.1.0\nyarl==1.25.1\n'
CHECK_IMPORTS = 'import torch, torchaudio, gradio; from f5_tts.api import F5TTS; print("F5-TTS imports: OK")'
# Full dependency resolution from a pinned lock, including addict and phonemizers.
LOCK = WORK / 'requirements.lock'
LOCK.write_text(LOCK_CONTENT)
run_logged([str(UV), 'pip', 'sync', '--python', str(PY), '--no-cache',
            '--index', 'https://download.pytorch.org/whl/cu124',
            '--index', 'https://pypi.org/simple', '--index-strategy', 'unsafe-best-match', str(LOCK)])
run_logged([str(UV), 'pip', 'check', '--python', str(PY)])
run_logged([str(PY), '-c', CHECK_IMPORTS])
print('Complete installation + imports: OK')


In [ ]:
# Verify the actual AI runtime, not Colab's preinstalled torch.
run_logged([str(PY), '-c', "import torch; print('Torch:', torch.__version__); print('CUDA:', torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'); assert torch.cuda.is_available(), 'Select T4 GPU in Runtime > Change runtime type, then rerun.'"])
print('T4/CUDA check: OK')


In [ ]:
# Write the genuine model application; no GitHub code download needed.
APP = WORK / 'app.py'
APP.write_text("from pathlib import Path\nimport os\nOUT = Path(os.environ['AI_MUSIC_OUT'])\nVOICE_BANK = Path(os.environ['AI_MUSIC_VOICE_BANK'])\nimport gradio as gr\nimport soundfile as sf\nfrom pathlib import Path\nimport shutil, time, subprocess\n\n_tts = None\ndef get_tts():\n    global _tts\n    if _tts is None:\n        from f5_tts.api import F5TTS\n        print('⏳ Loading F5-TTS (first run downloads ~1.3 GB to Drive)...')\n        _tts = F5TTS()\n    return _tts\n\ndef profiles():\n    return sorted(p.stem for p in VOICE_BANK.glob('*.wav'))\n\ndef save_voice(sample, profile_name, ref_text):\n    if sample is None:\n        raise gr.Error('Upload a voice sample first (5-15 seconds of clear speech works best).')\n    safe = ''.join(c for c in (profile_name or '') if c.isalnum() or c in (' ', '_', '-')).strip()\n    if not safe:\n        raise gr.Error('Enter a profile name.')\n    dst = VOICE_BANK / f'{safe}.wav'\n    # Convert to mono 24 kHz WAV and trim to 15 s (F5-TTS works best with short references)\n    subprocess.run(['ffmpeg', '-y', '-v', 'error', '-i', str(sample), '-ac', '1', '-ar', '24000', '-t', '15', str(dst)], check=True)\n    (VOICE_BANK / f'{safe}.txt').write_text((ref_text or '').strip())\n    return f'Saved: {dst}', gr.update(choices=profiles(), value=safe)\n\ndef generate(text, profile_name, speed):\n    if not (text or '').strip():\n        raise gr.Error('Enter text.')\n    if not profile_name or profile_name not in profiles():\n        raise gr.Error('Choose a saved voice profile.')\n    ref = VOICE_BANK / f'{profile_name}.wav'\n    if not profile_name or not ref.exists():\n        raise gr.Error('Choose a saved voice profile.')\n    ref_txt = VOICE_BANK / f'{profile_name}.txt'\n    ref_text = ref_txt.read_text() if ref_txt.exists() else ''  # empty = auto-transcribe with Whisper\n    out = OUT / f'{profile_name}_{time.time_ns()}.wav'\n    get_tts().infer(ref_file=str(ref), ref_text=ref_text, gen_text=text, file_wave=str(out), speed=float(speed))\n    return str(out)\n\nwith gr.Blocks(title='AI Music OS — Voice Clone') as demo:\n    gr.Markdown('## 🎙️ Voice Clone (F5-TTS, free & local)\\nUse only voices you have permission to clone.')\n    with gr.Tab('1. Save a voice'):\n        sample = gr.Audio(type='filepath', label='Reference voice sample (5-15 s)')\n        ref_text = gr.Textbox(label='What is said in the sample (optional — leave empty to auto-detect)')\n        profile = gr.Textbox(label='Profile name')\n        save = gr.Button('Save profile', variant='primary')\n        status = gr.Textbox(label='Status')\n    with gr.Tab('2. Speak with it'):\n        pick = gr.Dropdown(profiles(), label='Voice profile')\n        text = gr.Textbox(lines=6, label='Text to speak')\n        speed = gr.Slider(0.7, 1.3, value=1.0, step=0.05, label='Speed')\n        gen = gr.Button('Generate', variant='primary')\n        out = gr.Audio(label='Cloned speech', type='filepath')\n    save.click(save_voice, [sample, profile, ref_text], [status, pick])\n    gen.click(generate, [text, pick, speed], out)\n\ndemo.queue().launch(share=True, server_name='0.0.0.0', server_port=7867, allowed_paths=[str(OUT)])\n")
ENV['AI_MUSIC_OUT'] = str(OUT)
ENV['AI_MUSIC_VOICE_BANK'] = str(VOICE_BANK)
COMMAND = [str(PY), str(APP)]


In [ ]:
# Launch in the isolated environment; rerunning safely replaces the previous process.
if 'proc' in globals() and proc.poll() is None:
    proc.terminate()
    try: proc.wait(timeout=20)
    except subprocess.TimeoutExpired: proc.kill(); proc.wait()
LAUNCH_LOG = OUT / 'launch_v5.log'
with LAUNCH_LOG.open('w') as log:
    proc = subprocess.Popen(COMMAND, cwd=str(OUT), env=ENV, stdout=log, stderr=subprocess.STDOUT)
url = None
for _ in range(900):
    time.sleep(2)
    text = LAUNCH_LOG.read_text(errors='replace')
    if proc.poll() is not None:
        print(text[-18000:])
        raise RuntimeError(f'Application exited. Full log: {LAUNCH_LOG}')
    match = re.search(r'https://[a-zA-Z0-9-]+\.gradio\.live', text)
    if match: url = match.group(0); break
if url is None:
    print(LAUNCH_LOG.read_text(errors='replace')[-6000:])
    raise TimeoutError(f'No share URL after 30 minutes; inspect {LAUNCH_LOG}. Rerun this cell to retry.')
print('Open Music OS:', url)
print('Keep this Colab session open. Logs:', LAUNCH_LOG)


In [ ]:
# Inspect the application log at any time.
print(LAUNCH_LOG.read_text(errors='replace')[-15000:])
